# 🌍 07 · External Validation — denovo-db

> **Chapter 7.** The single most important experiment in the
> project: does a ClinVar-trained classifier actually generalize to
> a dataset it wasn't trained on?

## What denovo-db is

`denovo-db` is a curated collection of *de-novo* variants observed in
sequencing studies of affected individuals (autism, developmental
disorder, epilepsy, congenital heart disease, schizophrenia) and
documented sibling controls.

For our purposes:

* **Pathogenic label (y=1)** — variants in affected probands with a
  disease phenotype in `{autism, developmentalDisorder,
  intellectualDisability, epilepsy, schizophrenia,
  congenitalHeartDisease}`.
* **Benign label (y=0)** — variants in documented controls
  (`siblingcontrol`, `unaffected`).
* **Dropped rows** — phenotype ∉ {pathogenic, control}, or non-missense
  `FunctionClass`, or unresolvable coordinates. We report dropped-row
  counts openly in `results/metrics/external_denovo_db_unmapped.csv`.

**Why this is harder than ClinVar.** Almost every missense variant
in denovo-db — affected or control — looks damaging at the variant
level (high conservation, disruptive substitution). Separating them
requires *gene-level context*: "is this gene known to cause disease
when damaged?"

---

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path as _Path
sys.path.insert(0, str(_Path.cwd().parent))
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

REPO = Path.cwd().parent
metrics = pd.read_csv(REPO / "results/metrics/external_denovo_db_metrics.csv")
cov = pd.read_csv(REPO / "results/metrics/external_denovo_db_coverage.csv")

print("denovo-db coverage:")
print(cov.to_string(index=False))

## Current results (post gnomAD constraint features)

The numbers below are **after Phase 2 step 1**, which added gnomAD
gene-level constraint (`pLI`, `LOEUF`, `mis_z`, `oe_mis_upper`,
`lof_z`) to the model. The constraint features were imputed with
train-fit medians on 3.7 % of rows (documented).

| Slice | n | n_pos | ROC-AUC (95% CI) | PR-AUC (95% CI) |
|---|---:|---:|---|---|

In [ ]:
for _, r in metrics.iterrows():
    print(f"  {r['slice']:>22s}  n={int(r['n']):>4d}  n_pos={int(r['n_pos']):>4d}  "
          f"ROC {r['roc_auc']:.3f} [{r['roc_auc_ci_lo']:.3f}, {r['roc_auc_ci_hi']:.3f}]  "
          f"PR {r['pr_auc']:.3f} [{r['pr_auc_ci_lo']:.3f}, {r['pr_auc_ci_hi']:.3f}]")

## Before vs after the constraint features

| Slice | ROC pre | ROC post | PR pre | PR post |
|---|---:|---:|---:|---:|
| full | 0.468 | 0.511 | 0.761 | 0.790 |
| **family-holdout only** | **0.487** | **0.573** | **0.789** | **0.838** |

The family-holdout slice is the headline: for gene families the
model never saw during training, ROC jumped **from 0.487 (chance)
to 0.573** — a +0.086 move driven entirely by gene-level priors. It
is exactly the regime where only gene-level features can help.

---

## Family-holdout slice — why it's the honest number

`denovo_db_family_holdout` includes only variants whose gene family
is absent from the training split. It's the slice reviewers trust
most because it can't be inflated by paralog leakage.

Computation: for each denovo-db variant, compute its gene family via
`src.data_splitting.assign_gene_family`, then intersect with the
training family set. Rows whose family is not in train form the
holdout slice.

In [ ]:
# Per-slice scatterplot of predictions.
preds = pd.read_parquet(REPO / "results/metrics/external_denovo_db_predictions.parquet")
fig, (ax_full, ax_hold) = plt.subplots(1, 2, figsize=(11, 4))
for ax, title, df in [(ax_full, "full", preds),
                      (ax_hold, "family_holdout_only", preds[preds["family_holdout"]])]:
    for lbl, color in [(0, "#2ecc71"), (1, "#e74c3c")]:
        sub = df[df["label"] == lbl]
        ax.hist(sub["p_calibrated"], bins=25, alpha=0.6, color=color,
                label="benign" if lbl == 0 else "pathogenic")
    ax.set_xlabel("Calibrated probability")
    ax.set_ylabel("Variant count")
    ax.set_title(f"{title}  (n={len(df)}, {int(df['label'].sum())} pathogenic)")
    ax.legend(loc="upper left", frameon=False)
fig.suptitle("denovo-db — prediction distributions by slice", fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## What's still needed to close the gap

The family-holdout ROC at 0.573 is above chance but far from useful.
Closing the remaining gap will require phenotype-specific gene priors
(gene-disease association scores), which live outside the tabular
baseline's scope. That's the roadmap for Phase 2:

1. **ESM-2 zero-shot LLR** (Phase 2.1, underway on Colab)
2. **AlphaFold2 structural features** (Phase 2.2)
3. **ProteinGym DMS** as a *third* external validation source.

See notebook 09 for the ESM-2 zero-shot proof-of-concept — the
infrastructure already works on denovo-db, and rank-fusion with
XGBoost gave a small but real additive gain (+0.015 ROC on
holdout).

---

## Reproduce

```bash
python scripts/evaluate_external.py --only denovo_db --use-vep \
    --sample 1000 --n-boot 1000
```

Artifacts:
* `results/metrics/external_denovo_db_metrics.csv`
* `results/metrics/external_denovo_db_coverage.csv`
* `results/metrics/external_denovo_db_predictions.parquet`
* `results/metrics/external_denovo_db_unmapped.csv` (rows that
  couldn't be featurized; reported openly)